# Chapter 17: The Complex Projective Line

**Source orientation.** Sections 17.1-17.7; printed pages 311-328; PDF pages 333-350. The source span was read for structure, terminology, and concept coverage only. The prose, examples, code, diagrams, and checks here are original.

**Chapter goal.** Build a working model of `CP1` in which one extra point at infinity makes complex numbers, circles, lines, Mobius maps, inversions, cross-ratio angle tests, Grassmann-Plucker algebra, and stereographic projection part of one geometry.

**Chapter question.** How does adding the point at infinity turn circles, lines, inversions, and sphere geometry into one picture?

The chapter is best read as a sequence of translations. Homogeneous coordinates tell us what infinity means. Cross-ratios tell us when a visual circle or line is a projective fact. Mobius transformations preserve those facts. Anti-Mobius maps preserve the same circle family while reversing oriented sidedness. The Grassmann-Plucker relation explains why a projective determinant identity casts a Euclidean shadow as Ptolemy's theorem. Stereographic projection closes the loop by showing `CP1` as a sphere.

In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives-on-Projective-Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-17-the-complex-projective-line"
ARTIFACT_ROOT

## Computational Translation Guide

| Source idea | Computational representation | What to inspect |
| --- | --- | --- |
| `CP1` | nonzero vectors `(z0, z1)` modulo nonzero complex scale | finite chart `z0 / z1`, infinity chart `z1 / z0`, and scale invariance |
| point at infinity | homogeneous point `(1, 0)` | large finite values become small in the reciprocal chart |
| collinearity in the complex plane | `(B - A) / (C - A)` has zero imaginary part | the quotient tests direction, not a projective invariant by itself |
| circle or line through four points | complex cross-ratio is real or infinite | cocircularity becomes a projective condition on `CP1` |
| Mobius transformation | matrix action followed by dehomogenization, `(a z + b) / (c z + d)` | cross-ratios, circles, and oriented circle sides are preserved |
| anti-Mobius reflection or inversion | conjugation followed by a Mobius map, for example `z -> conjugate(z)` or `z -> 1 / conjugate(z)` | circles remain circles, but the sign of the cross-ratio side test reverses |
| Grassmann-Plucker relation | `[AB][CD] - [AC][BD] + [AD][BC] = 0` in two homogeneous coordinates | taking absolute values gives the Ptolemy inequality |
| intersection angle | argument of a cross-ratio for two oriented circles | Mobius maps preserve the signed angle modulo `2*pi` |
| stereographic projection | rational map from `C union {infinity}` to a sphere tangent to the plane | infinity becomes the north pole and circles stay circles |

**Library routing.** NumPy handles numerical complex arithmetic and samples. Matplotlib is used for durable labeled 2D diagrams and static 3D fallbacks. Plotly is used for interactive Mobius and stereographic views because the geometry benefits from pan, zoom, and rotation. SymPy is used for the exact Grassmann-Plucker identity. The course artifact helper records book-local paths and final sanity checks.

In [ ]:
import math
import cmath
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import sympy as sp
from matplotlib.patches import Arc, Circle, FancyArrowPatch

from utils.artifacts import (
    assert_artifacts,
    artifact_path,
    book_relative,
    display_artifact,
    ensure_artifact_root,
    save_json,
    save_table,
)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})
ensure_artifact_root(ARTIFACT_ROOT)

artifact_paths = []
image_paths = []
html_paths = []
table_paths = []
checks = {}


def figure_path(filename: str) -> Path:
    return artifact_path(ARTIFACT_ROOT, "figures", filename)


def html_path(filename: str) -> Path:
    return artifact_path(ARTIFACT_ROOT, "html", filename)


def record_artifact(path: Path, kind: str) -> Path:
    path = Path(path)
    artifact_paths.append(path)
    if kind == "image":
        image_paths.append(path)
    elif kind == "html":
        html_paths.append(path)
    elif kind == "table":
        table_paths.append(path)
    return path


def rel(path: Path) -> str:
    return book_relative(path)


def cp1_point(z: complex | float | int | str) -> np.ndarray:
    if z == "inf":
        return np.array([1 + 0j, 0 + 0j], dtype=complex)
    return np.array([complex(z), 1 + 0j], dtype=complex)


def bracket2(p: np.ndarray, q: np.ndarray) -> complex:
    return complex(p[0] * q[1] - p[1] * q[0])


def dehomogenize(p: np.ndarray, tol: float = 1e-12) -> complex | str:
    if abs(p[1]) < tol:
        return "inf"
    return complex(p[0] / p[1])


def cp1_cross_ratio(a, b, c, d) -> complex:
    A, B, C, D = map(cp1_point, [a, b, c, d])
    return (bracket2(A, C) * bracket2(B, D)) / (bracket2(A, D) * bracket2(B, C))


def mobius(z, a, b, c, d, tol: float = 1e-12):
    if z == "inf":
        if abs(c) < tol:
            return "inf"
        return a / c
    z = complex(z)
    den = c * z + d
    if abs(den) < tol:
        return "inf"
    return (a * z + b) / den


def anti_unit_inversion(z, tol: float = 1e-12):
    z = complex(z)
    if abs(z) < tol:
        return "inf"
    return 1 / z.conjugate()


def finite_mobius_array(zs, a, b, c, d, cutoff=8.0):
    out = []
    for z in zs:
        w = mobius(z, a, b, c, d)
        if w == "inf" or abs(w) > cutoff:
            out.append(np.nan + 1j * np.nan)
        else:
            out.append(w)
    return np.array(out, dtype=complex)


def plot_complex_axes(ax, limit=3.0):
    ax.axhline(0, color="0.75", lw=0.8)
    ax.axvline(0, color="0.75", lw=0.8)
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.grid(True, color="0.92", linewidth=0.6)


def circle_from_three(a: complex, b: complex, c: complex):
    x1, y1 = a.real, a.imag
    x2, y2 = b.real, b.imag
    x3, y3 = c.real, c.imag
    mat = np.array([[2 * (x2 - x1), 2 * (y2 - y1)], [2 * (x3 - x1), 2 * (y3 - y1)]], dtype=float)
    rhs = np.array([
        x2 * x2 + y2 * y2 - x1 * x1 - y1 * y1,
        x3 * x3 + y3 * y3 - x1 * x1 - y1 * y1,
    ], dtype=float)
    center_xy = np.linalg.solve(mat, rhs)
    center = complex(center_xy[0], center_xy[1])
    radius = abs(a - center)
    return center, radius


def fit_circle(zs):
    zs = np.asarray(zs, dtype=complex)
    xs, ys = zs.real, zs.imag
    mat = np.column_stack([2 * xs, 2 * ys, np.ones_like(xs)])
    rhs = xs * xs + ys * ys
    cx, cy, k = np.linalg.lstsq(mat, rhs, rcond=None)[0]
    radius_sq = cx * cx + cy * cy + k
    radius = math.sqrt(max(radius_sq, 0.0))
    residual = np.max(np.abs((xs - cx) ** 2 + (ys - cy) ** 2 - radius ** 2))
    return complex(cx, cy), radius, float(residual)


def oriented_side(a, b, c, p) -> float:
    return float(np.imag(cp1_cross_ratio(a, b, c, p)))


def angle_normalize(theta: float) -> float:
    return float((theta + math.pi) % (2 * math.pi) - math.pi)


def linearly_spaced_complex(start, stop, n=160):
    t = np.linspace(0, 1, n)
    return start + t * (stop - start)


def image_stats_relative(path: Path) -> dict:
    from PIL import Image

    image = Image.open(path).convert("RGB")
    arr = np.asarray(image, dtype=float)
    return {
        "path": rel(path),
        "width": int(image.width),
        "height": int(image.height),
        "pixel_std": float(arr.std()),
        "file_size": int(path.stat().st_size),
    }


## Refreshed Visual Storyboard

The previous scaffold compressed the chapter into generic route artifacts. This notebook replaces that scaffold with direct constructions tied to the chapter's seven sections. Every visual below has an inspection target and a check.

1. `CP1` charts and infinity: see how the finite chart and reciprocal chart overlap.
2. Geometric property tests: compare a direction quotient with the projective cross-ratio test for circles and lines.
3. Mobius transformations: watch a projective matrix act on a grid and on circles while preserving cross-ratio.
4. Inversions and Mobius reflections: separate orientation-preserving Mobius maps from anti-Mobius sidedness reversal.
5. Grassmann-Plucker and Ptolemy: turn a bracket identity into a Euclidean length inequality.
6. Intersection angles: read the angle between oriented circles from the argument of a cross-ratio.
7. Stereographic projection: identify `CP1` with a sphere and send infinity to the north pole.
8. Applied lab: determine a Mobius transformation from three point correspondences and verify what survives.

In [ ]:
storyboard = {
    "chapter goal": "Model CP1 as the common language for complex numbers, infinity, circles, Mobius maps, inversions, angles, and stereographic projection.",
    "source span read": "sections 17.1-17.7; printed pages 311-328; PDF pages 333-350",
    "concept inventory": [
        "CP1 as nonzero complex pairs modulo nonzero complex scale",
        "finite chart z=a/b and reciprocal chart w=b/a near infinity",
        "quotient tests for direction and collinearity in the finite complex plane",
        "real cross-ratio test for cocircularity or collinearity in CP1",
        "Mobius transformations as projective transformations of CP1",
        "orientation-preserving circle sidedness under Mobius maps",
        "anti-Mobius conjugation and unit-circle inversion reverse sidedness",
        "rank-2 Grassmann-Plucker relation and Ptolemy's theorem",
        "intersection angle of oriented circles as the argument of a cross-ratio",
        "stereographic projection from C union infinity to a tangent sphere",
    ],
    "library routing table": [
        {"concept": "CP1 charts", "representation": "two chart diagram", "library": "Matplotlib, NumPy", "why": "static labeled chart comparison is the clearest inspection target", "fallback": "plain coordinate table"},
        {"concept": "circle/line property tests", "representation": "cross-ratio side heatmap and quotient line test", "library": "Matplotlib, NumPy", "why": "the sign and zero set are visual scalar fields", "fallback": "sampled numeric table"},
        {"concept": "Mobius maps", "representation": "grid and circle image with interactive HTML", "library": "Plotly plus Matplotlib", "why": "the map bends the grid and benefits from zooming", "fallback": "static PNG"},
        {"concept": "anti-Mobius inversion/reflection", "representation": "unit-circle inversion arrows and side-sign samples", "library": "Matplotlib, NumPy", "why": "inside/outside exchange is a planar visual", "fallback": "before/after table"},
        {"concept": "Grassmann-Plucker to Ptolemy", "representation": "quadrilateral plus complex vector triangle", "library": "SymPy, Matplotlib", "why": "needs exact identity and visible Euclidean shadow", "fallback": "symbolic equality only"},
        {"concept": "intersection angle", "representation": "two oriented circles, tangents, angle arc", "library": "Matplotlib, NumPy", "why": "the cross-ratio argument can be compared to tangent directions", "fallback": "numeric tangent calculation"},
        {"concept": "stereographic projection", "representation": "3D sphere plus interactive rotation", "library": "Plotly, Matplotlib, NumPy", "why": "CP1 as sphere is spatial and should rotate", "fallback": "static 3D PNG"},
    ],
    "visual sequence": [
        {"concept": "CP1 charts/infinity", "artifact_filename": "cp1-two-charts-infinity.png", "inspection_target": "large finite z values collapse near w=0 in the infinity chart", "validation": "homogeneous rescaling leaves dehomogenized z unchanged"},
        {"concept": "geometric property tests", "artifact_filename": "cross-ratio-circle-line-tests.png", "inspection_target": "the circle boundary is the zero contour of imaginary cross-ratio", "validation": "line quotient and cocircular cross-ratio have zero imaginary part"},
        {"concept": "Mobius projective transformation", "artifact_filename": "mobius-grid-circle-action.html", "inspection_target": "grid lines and circles map to generalized circles", "validation": "cross-ratio residual and fitted-circle residual are small"},
        {"concept": "inversions and Mobius reflections", "artifact_filename": "mobius-reflection-inversion-sides.png", "inspection_target": "conjugation and unit inversion reverse side sign", "validation": "inversion is an involution and fixes unit circle samples"},
        {"concept": "Grassmann-Plucker/Ptolemy", "artifact_filename": "grassmann-plucker-ptolemy-shadow.png", "inspection_target": "complex bracket vector equality casts a length inequality shadow", "validation": "SymPy identity is zero and cyclic rectangle has zero Ptolemy gap"},
        {"concept": "intersection angles", "artifact_filename": "oriented-circle-intersection-angle.png", "inspection_target": "cross-ratio argument matches the signed tangent angle", "validation": "angle is preserved by a Mobius map"},
        {"concept": "stereographic projection", "artifact_filename": "stereographic-projection-cp1-sphere.html", "inspection_target": "finite complex points land on the tangent sphere and infinity becomes the north pole", "validation": "sphere equation and inverse projection residuals are small"},
        {"concept": "three-point Mobius lab", "artifact_filename": "three-point-mobius-lab.png", "inspection_target": "three correspondences determine the matrix up to scale", "validation": "correspondence residual and cross-ratio residual are small"},
    ],
    "artifact plan": {
        "figures": [
            "artifacts/chapter-17-the-complex-projective-line/figures/cp1-two-charts-infinity.png",
            "artifacts/chapter-17-the-complex-projective-line/figures/cross-ratio-circle-line-tests.png",
            "artifacts/chapter-17-the-complex-projective-line/figures/mobius-grid-circle-action.png",
            "artifacts/chapter-17-the-complex-projective-line/figures/mobius-reflection-inversion-sides.png",
            "artifacts/chapter-17-the-complex-projective-line/figures/grassmann-plucker-ptolemy-shadow.png",
            "artifacts/chapter-17-the-complex-projective-line/figures/oriented-circle-intersection-angle.png",
            "artifacts/chapter-17-the-complex-projective-line/figures/stereographic-projection-cp1-sphere.png",
            "artifacts/chapter-17-the-complex-projective-line/figures/three-point-mobius-lab.png",
        ],
        "html": [
            "artifacts/chapter-17-the-complex-projective-line/html/mobius-grid-circle-action.html",
            "artifacts/chapter-17-the-complex-projective-line/html/stereographic-projection-cp1-sphere.html",
        ],
        "tables": ["artifacts/chapter-17-the-complex-projective-line/tables/chapter-17-check-summary.csv"],
        "checks": [
            "artifacts/chapter-17-the-complex-projective-line/checks/storyboard.json",
            "artifacts/chapter-17-the-complex-projective-line/checks/visual-checks.json",
            "artifacts/chapter-17-the-complex-projective-line/checks/final-sanity.json",
        ],
    },
    "computational checks": [
        "homogeneous rescaling in CP1",
        "collinearity quotient and cocircular cross-ratio realness",
        "Mobius cross-ratio invariance and transformed-circle residual",
        "anti-Mobius side-sign reversal and unit inversion involution",
        "exact Grassmann-Plucker identity and Ptolemy equality case",
        "cross-ratio angle equals tangent angle and is Mobius invariant",
        "stereographic points satisfy the sphere equation and round-trip inverse",
        "artifact existence, size, image variation, and book-local JSON paths",
    ],
    "proof-visualization strategy": "Use scalar-field zero sets, vector-triangle identities, exact symbolic brackets, and tangent-angle comparisons instead of copied textbook proof text.",
    "implementation notes": "Notebook is direct chapter code; no shared utilities or indexes were edited. Generated paths are book-local under the Chapter 17 artifact subtree.",
    "risks": "Plotly HTML is saved as an artifact, while static PNG fallbacks support headless validation. The source PDF is used only for orientation.",
    "acceptance criteria": "Notebook executes with nbclient/validate_ppg_course chapter 17; scoped notebook and visual audits pass; git diff --check passes for assigned files.",
}
record_artifact(save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json"), "check")
storyboard["visual sequence"]


## 1. Two Charts for `CP1`

A finite complex number appears as the homogeneous class of `(z, 1)`. The point at infinity appears as `(1, 0)`. The useful mental model is not that infinity is a distant pixel, but that `CP1` needs two overlapping charts. The usual finite chart reads `z = z0 / z1`; the reciprocal chart reads `w = z1 / z0 = 1 / z` and places infinity at `w = 0`.

Inspect the figure from left to right: circles with large `|z|` in the finite chart become tiny circles around the origin in the infinity chart.

In [ ]:
theta = np.linspace(0, 2 * math.pi, 500)
radii = [0.5, 1, 2, 4, 8]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), constrained_layout=True)

ax = axes[0]
plot_complex_axes(ax, 5)
for r in radii[:-1]:
    ax.plot(r * np.cos(theta), r * np.sin(theta), color="#4C78A8", lw=1.1, alpha=0.75)
for angle in np.linspace(0, 2 * math.pi, 8, endpoint=False):
    ax.plot([0, 4.6 * math.cos(angle)], [0, 4.6 * math.sin(angle)], color="#C7D3E3", lw=0.8)
for z, label in [(1 + 0.5j, "z"), (2.2 + 1.2j, "2z"), (4.4 + 2.4j, "large z")]:
    ax.scatter([z.real], [z.imag], s=35, color="#E45756", zorder=4)
    ax.text(z.real + 0.12, z.imag + 0.12, label)
ax.annotate("finite chart z=z0/z1", xy=(3.2, 3.5), xytext=(0.7, 4.4), arrowprops={"arrowstyle": "->", "lw": 0.8})
ax.annotate("infinity is not inside this chart", xy=(4.7, 0), xytext=(1.2, -4.3), arrowprops={"arrowstyle": "->", "lw": 0.8})
ax.set_title("Finite chart: C")

ax = axes[1]
plot_complex_axes(ax, 2.3)
for r in radii:
    wr = 1 / r
    ax.plot(wr * np.cos(theta), -wr * np.sin(theta), color="#59A14F", lw=1.1, alpha=0.75)
for z in [1 + 0.5j, 2.2 + 1.2j, 4.4 + 2.4j]:
    w = 1 / z
    ax.scatter([w.real], [w.imag], s=35, color="#E45756", zorder=4)
    ax.text(w.real + 0.05, w.imag + 0.05, f"1/({z.real:.1f}+{z.imag:.1f}i)", fontsize=8)
ax.scatter([0], [0], s=60, color="#F28E2B", edgecolor="black", zorder=5)
ax.text(0.07, 0.08, "infinity: w=0")
ax.annotate("large |z| becomes small |w|", xy=(0.14, -0.07), xytext=(-1.8, -1.7), arrowprops={"arrowstyle": "->", "lw": 0.8})
ax.set_title("Infinity chart: w=1/z")

path = record_artifact(figure_path("cp1-two-charts-infinity.png"), "image")
fig.savefig(path, bbox_inches="tight")
plt.close(fig)

z_sample = 1.25 - 0.5j
scale = 2 - 3j
rescaled = dehomogenize(scale * cp1_point(z_sample))
checks["cp1_charts"] = {
    "homogeneous_rescale_error": float(abs(rescaled - z_sample)),
    "reciprocal_large_radius": float(abs(1 / (80 - 10j))),
    "infinity_dehomogenizes_to": dehomogenize(cp1_point("inf")),
}
display_artifact(path, width=860)
checks["cp1_charts"]


## 2. Geometric Property Tests

In the finite complex plane, the quotient `(B - A) / (C - A)` tests whether the directions from `A` to `B` and `A` to `C` agree up to sign. That is useful but still a Euclidean-plane test.

The projective test in this chapter is stronger: four points of `CP1` lie on a common circle or line exactly when their cross-ratio is real or infinite. For an oriented circle `(A, B, C)`, the imaginary part of `(A, B; C, p)` is a side predicate: zero on the boundary, one sign on one side, the opposite sign on the other.

In [ ]:
line_A = -1.4 - 0.6j
line_B = 0.15 + 0.05j
line_C = line_A + 2.0 * (line_B - line_A)
line_Q = (line_B - line_A) / (line_C - line_A)

A, B, C, D = 1 + 0j, -1 + 0j, 0 + 1j, 0 - 1j
grid = np.linspace(-1.8, 1.8, 260)
xx, yy = np.meshgrid(grid, grid)
zz = xx + 1j * yy
side_values = np.empty_like(xx)
for i in range(xx.shape[0]):
    for j in range(xx.shape[1]):
        p = zz[i, j]
        if min(abs(p - A), abs(p - B), abs(p - C)) < 1e-6:
            side_values[i, j] = np.nan
        else:
            side_values[i, j] = oriented_side(A, B, C, p)
side_values = np.clip(side_values, -3, 3)

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.8), constrained_layout=True)
ax = axes[0]
plot_complex_axes(ax, 2.0)
line_pts = linearly_spaced_complex(line_A - 0.3 * (line_C - line_A), line_C + 0.3 * (line_C - line_A), 20)
ax.plot(line_pts.real, line_pts.imag, color="#4C78A8", lw=2)
ax.scatter([line_A.real, line_B.real, line_C.real], [line_A.imag, line_B.imag, line_C.imag], color="#E45756", s=45, zorder=4)
for label, z in [("A", line_A), ("B", line_B), ("C", line_C)]:
    ax.text(z.real + 0.05, z.imag + 0.08, label, weight="bold")
off = 0.2 + 1.1j
ax.scatter([off.real], [off.imag], color="#F28E2B", s=45, zorder=4)
ax.text(off.real + 0.05, off.imag + 0.08, "off line")
ax.set_title("Collinearity: quotient has zero imaginary part")
ax.text(-1.9, -1.75, f"Im((B-A)/(C-A)) = {line_Q.imag:.2e}", fontsize=9)

ax = axes[1]
plot_complex_axes(ax, 1.9)
mesh = ax.contourf(xx, yy, side_values, levels=np.linspace(-3, 3, 17), cmap="RdBu_r", alpha=0.75)
ax.contour(xx, yy, side_values, levels=[0], colors="black", linewidths=2)
ax.add_patch(Circle((0, 0), 1, fill=False, color="black", lw=1, ls="--"))
ax.scatter([A.real, B.real, C.real, D.real], [A.imag, B.imag, C.imag, D.imag], color="#E45756", s=35, zorder=5)
for label, z in [("A", A), ("B", B), ("C", C), ("D", D)]:
    ax.text(z.real + 0.05, z.imag + 0.05, label, weight="bold")
ax.scatter([0.25], [0.15], color="#F28E2B", s=38, zorder=5)
ax.text(0.32, 0.20, "p")
ax.set_title("Cocircularity: Im cross-ratio zero contour")
fig.colorbar(mesh, ax=ax, shrink=0.84, label="Im(A,B;C,p)")

path = record_artifact(figure_path("cross-ratio-circle-line-tests.png"), "image")
fig.savefig(path, bbox_inches="tight")
plt.close(fig)

circle_cr = cp1_cross_ratio(A, B, C, D)
off_side = oriented_side(A, B, C, 0.25 + 0.15j)
checks["property_tests"] = {
    "collinearity_quotient_imag": float(abs(line_Q.imag)),
    "cocircular_cross_ratio_imag": float(abs(circle_cr.imag)),
    "cocircular_cross_ratio_real": float(circle_cr.real),
    "off_circle_side_abs": float(abs(off_side)),
}
display_artifact(path, width=900)
checks["property_tests"]


## 3. Mobius Transformations as Projective Transformations

A nondegenerate complex matrix acts on homogeneous coordinates, then the finite chart reads the result as `(a z + b) / (c z + d)`. This is the Mobius transformation. The matrix is only defined up to nonzero scalar scale, which is why the map has three complex degrees of freedom.

The artifact below maps a square grid and a circle. Grid lines usually become arcs, but the projective statement remains simple: generalized circles, meaning finite circles or lines through infinity, map to generalized circles. The validation checks a cross-ratio before and after the map and fits a circle to the image of a sampled circle.

In [ ]:
a = 0.88 * cmath.exp(0.42j)
b = 0.48 - 0.28j
c = 0.18 + 0.12j
d = 1.0 - 0.08j
determinant = a * d - b * c

lines = []
for x in np.linspace(-2, 2, 9):
    lines.append(linearly_spaced_complex(x - 2.2j, x + 2.2j, 220))
for y in np.linspace(-2, 2, 9):
    lines.append(linearly_spaced_complex(-2.2 + 1j * y, 2.2 + 1j * y, 220))
unit_circle = 0.25 + 0.95 * np.exp(1j * theta)
image_lines = [finite_mobius_array(line, a, b, c, d, cutoff=7) for line in lines]
image_circle = finite_mobius_array(unit_circle, a, b, c, d, cutoff=7)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
ax = axes[0]
plot_complex_axes(ax, 2.6)
for line in lines:
    ax.plot(line.real, line.imag, color="#BAB0AC", lw=0.65)
ax.plot(unit_circle.real, unit_circle.imag, color="#E45756", lw=2, label="sample circle")
ax.set_title("Before: finite chart samples")
ax.legend(loc="upper right")

ax = axes[1]
plot_complex_axes(ax, 4.0)
for k, line in enumerate(image_lines):
    color = "#4C78A8" if k < 9 else "#59A14F"
    ax.plot(line.real, line.imag, color=color, lw=0.75, alpha=0.78)
ax.plot(image_circle.real, image_circle.imag, color="#E45756", lw=2, label="image of circle")
ax.set_title("After: z -> (a z + b)/(c z + d)")
ax.legend(loc="upper right")

path_png = record_artifact(figure_path("mobius-grid-circle-action.png"), "image")
fig.savefig(path_png, bbox_inches="tight")
plt.close(fig)

fig_html = go.Figure()
for k, line in enumerate(image_lines):
    fig_html.add_trace(go.Scatter(
        x=line.real,
        y=line.imag,
        mode="lines",
        line={"width": 1.2, "color": "#4C78A8" if k < 9 else "#59A14F"},
        name="vertical images" if k == 0 else ("horizontal images" if k == 9 else "grid image"),
        showlegend=k in {0, 9},
    ))
fig_html.add_trace(go.Scatter(x=image_circle.real, y=image_circle.imag, mode="lines", line={"width": 3, "color": "#E45756"}, name="circle image"))
fig_html.update_layout(
    title="Mobius action on a grid and circle",
    xaxis={"scaleanchor": "y", "title": "Re"},
    yaxis={"title": "Im"},
    template="plotly_white",
    width=780,
    height=620,
)
path_html = record_artifact(html_path("mobius-grid-circle-action.html"), "html")
fig_html.write_html(path_html, include_plotlyjs="cdn")

mobius_sample = [-1.2 + 0.4j, -0.15 + 1.1j, 0.85 - 0.55j, 1.65 + 0.25j]
mobius_image = [mobius(z, a, b, c, d) for z in mobius_sample]
cr_before = cp1_cross_ratio(*mobius_sample)
cr_after = cp1_cross_ratio(*mobius_image)
finite_image_circle = image_circle[np.isfinite(image_circle.real) & np.isfinite(image_circle.imag)]
_, _, circle_residual = fit_circle(finite_image_circle[::4])
checks["mobius_transform"] = {
    "matrix_determinant_abs": float(abs(determinant)),
    "cross_ratio_error": float(abs(cr_before - cr_after)),
    "transformed_circle_fit_residual": float(circle_residual),
}
display_artifact(path_png, width=900)
display_artifact(path_html, width="100%", height=520)
checks["mobius_transform"]


## 4. Inversions and Mobius Reflections

Mobius transformations preserve the sign of the imaginary part of a cross-ratio when the oriented circle and the test point are transformed together. Complex conjugation is different: it is a field automorphism followed by no projective matrix, so it preserves the circle family but reverses orientation.

The unit-circle inversion `z -> 1 / conjugate(z)` is anti-Mobius. It fixes the unit circle pointwise, sends the origin to infinity, and exchanges inside with outside. Algebraically, it also reverses the sign of the oriented side predicate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.8), constrained_layout=True)

ax = axes[0]
plot_complex_axes(ax, 2.2)
ax.add_patch(Circle((0, 0), 1, fill=False, color="black", lw=2))
for angle in np.linspace(0.25, 2 * math.pi - 0.35, 9):
    z = 0.45 * cmath.exp(1j * angle)
    w = anti_unit_inversion(z)
    ax.scatter([z.real], [z.imag], color="#4C78A8", s=26, zorder=3)
    ax.scatter([w.real], [w.imag], color="#E45756", s=26, zorder=3)
    ax.add_patch(FancyArrowPatch((z.real, z.imag), (w.real, w.imag), arrowstyle="->", mutation_scale=8, color="#666666", lw=0.8))
for angle in np.linspace(0, 2 * math.pi, 6, endpoint=False):
    u = cmath.exp(1j * angle)
    ax.scatter([u.real], [u.imag], color="#F28E2B", s=30, edgecolor="black", zorder=4)
ax.text(-2.05, 1.85, "blue inside -> red outside")
ax.set_title("Unit-circle inversion z -> 1/conjugate(z)")

ax = axes[1]
plot_complex_axes(ax, 2.2)
pts = np.array([-1.6 + 0.95j, -0.6 + 1.25j, 0.4 + 0.75j, 1.3 + 1.1j])
reflected = np.conjugate(pts)
ax.plot([-2, 2], [0, 0], color="black", lw=1.8)
ax.scatter(pts.real, pts.imag, color="#4C78A8", label="p")
ax.scatter(reflected.real, reflected.imag, color="#E45756", label="conjugate(p)")
for z, w in zip(pts, reflected):
    ax.plot([z.real, w.real], [z.imag, w.imag], color="#999999", lw=0.8, ls="--")
ax.text(-2.05, -1.9, "conjugation fixes the real-axis circle and flips sides")
ax.set_title("Anti-Mobius reflection")
ax.legend(loc="upper right")

path = record_artifact(figure_path("mobius-reflection-inversion-sides.png"), "image")
fig.savefig(path, bbox_inches="tight")
plt.close(fig)

side_A, side_B, side_C = 1 + 0j, 0 + 1j, -1 + 0j
side_p = 0.35 + 0.2j
side_original = oriented_side(side_A, side_B, side_C, side_p)
side_conjugated = oriented_side(side_A.conjugate(), side_B.conjugate(), side_C.conjugate(), side_p.conjugate())
side_inverted = oriented_side(
    anti_unit_inversion(side_A),
    anti_unit_inversion(side_B),
    anti_unit_inversion(side_C),
    anti_unit_inversion(side_p),
)
unit_samples = np.exp(1j * np.linspace(0, 2 * math.pi, 20, endpoint=False))
unit_fixed = max(abs(anti_unit_inversion(z) - z) for z in unit_samples)
involution_residual = max(abs(anti_unit_inversion(anti_unit_inversion(z)) - z) for z in [0.4 + 0.7j, -1.2 + 0.6j, 0.3 - 1.4j])
checks["anti_mobius"] = {
    "conjugation_side_sum": float(abs(side_original + side_conjugated)),
    "inversion_side_sum": float(abs(side_original + side_inverted)),
    "unit_circle_fixed_residual": float(unit_fixed),
    "inversion_involution_residual": float(involution_residual),
}
display_artifact(path, width=900)
checks["anti_mobius"]


## 5. Grassmann-Plucker Relations and Ptolemy's Shadow

In `CP1`, the two-coordinate bracket `[AB]` is a complex determinant. Four points satisfy the rank-2 Grassmann-Plucker relation

`[AB][CD] - [AC][BD] + [AD][BC] = 0`.

For finite points written as `(z, 1)`, the bracket is the complex difference `A - B`. The three summands are complex vectors. Taking absolute values and applying the triangle inequality produces Ptolemy's inequality. When the four points are cyclic in the right order, the vector equality has collinear summands and the inequality becomes equality.

In [ ]:
a0, a1, b0, b1, c0, c1, d0, d1 = sp.symbols("a0 a1 b0 b1 c0 c1 d0 d1")
A2, B2, C2, D2 = (a0, a1), (b0, b1), (c0, c1), (d0, d1)

def det_sym(P, Q):
    return P[0] * Q[1] - P[1] * Q[0]

expr = sp.expand(det_sym(A2, B2) * det_sym(C2, D2) - det_sym(A2, C2) * det_sym(B2, D2) + det_sym(A2, D2) * det_sym(B2, C2))
expr_simplified = sp.factor(expr)

quad = [0 + 0j, 3 + 0j, 3 + 4j, 0 + 4j]
Aq, Bq, Cq, Dq = quad
br = lambda x, y: x - y
alpha = br(Aq, Bq) * br(Cq, Dq)
beta = br(Aq, Dq) * br(Bq, Cq)
gamma = br(Aq, Cq) * br(Bq, Dq)
ptolemy_gap_rectangle = abs(Aq - Bq) * abs(Cq - Dq) + abs(Aq - Dq) * abs(Bq - Cq) - abs(Aq - Cq) * abs(Bq - Dq)
D_noncyclic = 0.6 + 4.2j
ptolemy_gap_noncyclic = abs(Aq - Bq) * abs(Cq - D_noncyclic) + abs(Aq - D_noncyclic) * abs(Bq - Cq) - abs(Aq - Cq) * abs(Bq - D_noncyclic)

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.7), constrained_layout=True)
ax = axes[0]
plot_complex_axes(ax, 4.6)
poly = np.array(quad + [quad[0]])
ax.plot(poly.real, poly.imag, color="#4C78A8", lw=2)
ax.plot([Aq.real, Cq.real], [Aq.imag, Cq.imag], color="#BAB0AC", lw=1.2, ls="--")
ax.plot([Bq.real, Dq.real], [Bq.imag, Dq.imag], color="#BAB0AC", lw=1.2, ls="--")
ax.scatter([z.real for z in quad], [z.imag for z in quad], color="#E45756", s=42, zorder=4)
for label, z in zip(["A", "B", "C", "D"], quad):
    ax.text(z.real + 0.1, z.imag + 0.1, label, weight="bold")
ax.text(-4.35, -4.05, "cyclic rectangle: 3*3 + 4*4 = 5*5")
ax.set_title("Ptolemy equality case")

ax = axes[1]
plot_complex_axes(ax, 28)
ax.add_patch(FancyArrowPatch((0, 0), (alpha.real, alpha.imag), arrowstyle="->", mutation_scale=12, color="#4C78A8", lw=2, label="alpha=[AB][CD]"))
ax.add_patch(FancyArrowPatch((alpha.real, alpha.imag), ((alpha + beta).real, (alpha + beta).imag), arrowstyle="->", mutation_scale=12, color="#59A14F", lw=2, label="beta=[AD][BC]"))
ax.add_patch(FancyArrowPatch((0, 0), (gamma.real, gamma.imag), arrowstyle="->", mutation_scale=12, color="#E45756", lw=1.8, ls="--", label="gamma=[AC][BD]"))
ax.scatter([0, alpha.real, gamma.real], [0, alpha.imag, gamma.imag], color="black", s=18)
ax.text(alpha.real + 1, alpha.imag + 1, "alpha")
ax.text(gamma.real + 1, gamma.imag + 1, "gamma=alpha+beta")
ax.set_title("Complex bracket vectors")
ax.legend(loc="lower left")

path = record_artifact(figure_path("grassmann-plucker-ptolemy-shadow.png"), "image")
fig.savefig(path, bbox_inches="tight")
plt.close(fig)

checks["grassmann_plucker_ptolemy"] = {
    "sympy_identity_zero": bool(expr_simplified == 0),
    "numeric_gp_residual": float(abs(alpha - gamma + beta)),
    "rectangle_ptolemy_gap": float(abs(ptolemy_gap_rectangle)),
    "noncyclic_ptolemy_gap_positive": float(ptolemy_gap_noncyclic),
}
display_artifact(path, width=900)
checks["grassmann_plucker_ptolemy"]


## 6. Intersection Angles from Cross-Ratios

Two oriented circles meeting at `P1` and `P2` can be encoded by auxiliary points `Q1` and `Q2`: `C1 = (P1, Q1, P2)` and `C2 = (P1, Q2, P2)`. The argument of `(Q1, Q2; P1, P2)` records the signed angle between their oriented tangents. Reversing either circle orientation reverses the sign.

The figure shows the circles, tangent arrows at `P1`, and the angle arc. The check then applies a Mobius map and confirms that the cross-ratio angle is unchanged.

In [ ]:
P1, P2 = 0 + 0j, 1 + 0j
Q1, Q2 = 0.35 + 0.85j, 0.62 - 0.55j

def ccw_passes(theta_start, theta_mid, theta_end):
    twopi = 2 * math.pi
    s = theta_start % twopi
    m = theta_mid % twopi
    e = theta_end % twopi
    if e < s:
        e += twopi
    if m < s:
        m += twopi
    return s <= m <= e


def oriented_tangent_at_start(p_start, q_mid, p_end):
    center, radius = circle_from_three(p_start, q_mid, p_end)
    th_start = cmath.phase(p_start - center)
    th_mid = cmath.phase(q_mid - center)
    th_end = cmath.phase(p_end - center)
    sign = 1 if ccw_passes(th_start, th_mid, th_end) else -1
    tangent = sign * 1j * (p_start - center)
    return tangent / abs(tangent), center, radius, sign

t1, center1, radius1, sign1 = oriented_tangent_at_start(P1, Q1, P2)
t2, center2, radius2, sign2 = oriented_tangent_at_start(P1, Q2, P2)
cr_angle = cmath.phase(cp1_cross_ratio(Q1, Q2, P1, P2))
tangent_angle = cmath.phase(t1 / t2)
angle_error = angle_normalize(cr_angle - tangent_angle)

fig, ax = plt.subplots(figsize=(7.2, 5.5), constrained_layout=True)
plot_complex_axes(ax, 1.75)
for center, radius, color, label in [(center1, radius1, "#4C78A8", "C1"), (center2, radius2, "#59A14F", "C2")]:
    ax.add_patch(Circle((center.real, center.imag), radius, fill=False, color=color, lw=2, label=label))
ax.scatter([P1.real, P2.real, Q1.real, Q2.real], [P1.imag, P2.imag, Q1.imag, Q2.imag], color="#E45756", s=42, zorder=5)
for label, z in [("P1", P1), ("P2", P2), ("Q1", Q1), ("Q2", Q2)]:
    ax.text(z.real + 0.04, z.imag + 0.05, label, weight="bold")
scale_arrow = 0.52
ax.add_patch(FancyArrowPatch((P1.real, P1.imag), ((P1 + scale_arrow * t1).real, (P1 + scale_arrow * t1).imag), arrowstyle="->", mutation_scale=12, color="#4C78A8", lw=2))
ax.add_patch(FancyArrowPatch((P1.real, P1.imag), ((P1 + scale_arrow * t2).real, (P1 + scale_arrow * t2).imag), arrowstyle="->", mutation_scale=12, color="#59A14F", lw=2))
arc_radius = 0.36
start_deg = math.degrees(cmath.phase(t2))
end_deg = math.degrees(cmath.phase(t1))
ax.add_patch(Arc((P1.real, P1.imag), 2 * arc_radius, 2 * arc_radius, angle=0, theta1=start_deg, theta2=end_deg, color="#E45756", lw=2))
ax.text(0.12, -0.42, f"arg cross-ratio = {cr_angle:.3f} rad")
ax.set_title("Oriented circle angle from a cross-ratio")
ax.legend(loc="upper right")

path = record_artifact(figure_path("oriented-circle-intersection-angle.png"), "image")
fig.savefig(path, bbox_inches="tight")
plt.close(fig)

angle_map_params = (1.05 + 0.2j, -0.2 + 0.35j, 0.18 - 0.07j, 0.95 + 0.1j)
P1m, P2m, Q1m, Q2m = [mobius(z, *angle_map_params) for z in [P1, P2, Q1, Q2]]
cr_angle_mapped = cmath.phase(cp1_cross_ratio(Q1m, Q2m, P1m, P2m))
checks["intersection_angles"] = {
    "cross_ratio_tangent_angle_error": float(abs(angle_error)),
    "mobius_angle_invariance_error": float(abs(angle_normalize(cr_angle - cr_angle_mapped))),
    "oriented_tangent_signs": [int(sign1), int(sign2)],
}
display_artifact(path, width=760)
checks["intersection_angles"]


## 7. Stereographic Projection: `CP1` as a Sphere

Place the complex plane at `z = 0` in three-dimensional space. Use the sphere with center `(0, 0, 1)` and radius `1`, tangent to the complex plane at the origin. Projection from the north pole `(0, 0, 2)` sends each finite complex number to the sphere, while values with very large modulus approach the north pole. The point at infinity completes the sphere.

This model explains why circles are the natural objects of `CP1`: finite lines become circles through the north pole, and finite circles become circles not necessarily through it.

In [ ]:
def stereo(z: complex) -> np.ndarray:
    x, y = z.real, z.imag
    r2 = x * x + y * y
    denom = r2 + 4
    return np.array([4 * x / denom, 4 * y / denom, 2 * r2 / denom], dtype=float)


def stereo_inverse(point: np.ndarray) -> complex:
    X, Y, Z = point
    factor = 2 / (2 - Z)
    return complex(factor * X, factor * Y)

sphere_u = np.linspace(0, 2 * math.pi, 80)
sphere_v = np.linspace(0, math.pi, 40)
SX = np.outer(np.cos(sphere_u), np.sin(sphere_v))
SY = np.outer(np.sin(sphere_u), np.sin(sphere_v))
SZ = 1 + np.outer(np.ones_like(sphere_u), np.cos(sphere_v))

plane_circle = 1.25 + 0.75 * np.exp(1j * theta)
plane_line = linearly_spaced_complex(-2.5 - 1.0j, 2.5 + 1.0j, 350)
proj_circle = np.array([stereo(z) for z in plane_circle])
proj_line = np.array([stereo(z) for z in plane_line])
rays = [0 + 0j, 1 + 0.8j, -1.4 + 0.6j, 2.4 - 0.8j]
ray_points = [stereo(z) for z in rays]
north = np.array([0.0, 0.0, 2.0])

fig = plt.figure(figsize=(8, 6), constrained_layout=True)
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(SX, SY, SZ, color="#D8E7F3", alpha=0.32, linewidth=0, shade=False)
ax.plot(proj_circle[:, 0], proj_circle[:, 1], proj_circle[:, 2], color="#E45756", lw=2.2, label="circle image")
ax.plot(proj_line[:, 0], proj_line[:, 1], proj_line[:, 2], color="#4C78A8", lw=2.0, label="line image")
for p in ray_points:
    ax.plot([north[0], p[0]], [north[1], p[1]], [north[2], p[2]], color="#999999", lw=0.8)
    ax.scatter([p[0]], [p[1]], [p[2]], color="#F28E2B", s=24)
ax.scatter([0], [0], [2], color="black", s=36)
ax.text(0.05, 0.05, 2.05, "infinity")
ax.set_title("Stereographic projection from C to the CP1 sphere")
ax.set_box_aspect((1, 1, 1))
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_zlim(0, 2.05)
ax.view_init(elev=23, azim=-62)
ax.legend(loc="upper left")
path_png = record_artifact(figure_path("stereographic-projection-cp1-sphere.png"), "image")
fig.savefig(path_png, bbox_inches="tight")
plt.close(fig)

fig_html = go.Figure()
fig_html.add_trace(go.Surface(x=SX, y=SY, z=SZ, opacity=0.35, colorscale=[[0, "#D8E7F3"], [1, "#D8E7F3"]], showscale=False, name="sphere"))
fig_html.add_trace(go.Scatter3d(x=proj_circle[:, 0], y=proj_circle[:, 1], z=proj_circle[:, 2], mode="lines", line={"color": "#E45756", "width": 5}, name="projected circle"))
fig_html.add_trace(go.Scatter3d(x=proj_line[:, 0], y=proj_line[:, 1], z=proj_line[:, 2], mode="lines", line={"color": "#4C78A8", "width": 5}, name="projected line"))
for p in ray_points:
    fig_html.add_trace(go.Scatter3d(x=[0, p[0]], y=[0, p[1]], z=[2, p[2]], mode="lines", line={"color": "#777777", "width": 2}, showlegend=False))
fig_html.add_trace(go.Scatter3d(x=[0], y=[0], z=[2], mode="markers+text", marker={"color": "black", "size": 5}, text=["infinity"], textposition="top center", name="north pole"))
fig_html.update_layout(
    title="Interactive stereographic projection model of CP1",
    scene={"xaxis_title": "X", "yaxis_title": "Y", "zaxis_title": "Z", "aspectmode": "cube"},
    template="plotly_white",
    width=780,
    height=650,
)
path_html = record_artifact(html_path("stereographic-projection-cp1-sphere.html"), "html")
fig_html.write_html(path_html, include_plotlyjs="cdn")

sample_zs = [0 + 0j, 0.8 - 0.6j, -1.3 + 1.7j, 3.2 - 0.5j]
sphere_residuals = []
roundtrip_residuals = []
for z in sample_zs:
    p = stereo(z)
    sphere_residuals.append(abs(p[0] ** 2 + p[1] ** 2 + (p[2] - 1) ** 2 - 1))
    roundtrip_residuals.append(abs(stereo_inverse(p) - z))
north_distance = float(np.linalg.norm(stereo(100 + 20j) - north))
checks["stereographic_projection"] = {
    "max_sphere_equation_residual": float(max(sphere_residuals)),
    "max_inverse_roundtrip_residual": float(max(roundtrip_residuals)),
    "large_value_distance_to_north_pole": north_distance,
}
display_artifact(path_png, width=760)
display_artifact(path_html, width="100%", height=560)
checks["stereographic_projection"]


## Applied Lab: Build a Mobius Map from Three Correspondences

A Mobius transformation is determined by three source-image pairs, provided the points are distinct and the target points are distinct. The lab solves the linear equations

`a z_i + b - w_i c z_i - w_i d = 0`

for the projective matrix `(a, b; c, d)`. Then it checks the three prescribed images, sends a test circle through the map, and compares a cross-ratio before and after. This is the computational version of the chapter's claim that `CP1` projective transformations have three complex degrees of freedom.

In [ ]:
def mobius_from_three(source, target):
    rows = []
    rhs = []
    for z, w in zip(source, target):
        rows.append([z, 1, -w * z])
        rhs.append(w)
    a0, b0, c0 = np.linalg.solve(np.asarray(rows, dtype=complex), np.asarray(rhs, dtype=complex))
    d0 = 1 + 0j
    norm = max(abs(a0), abs(b0), abs(c0), abs(d0))
    return a0 / norm, b0 / norm, c0 / norm, d0 / norm

source_pts = [-1.1 - 0.4j, 0.25 + 0.95j, 1.35 - 0.15j]
target_pts = [0.05 + 0.75j, -0.95 + 0.1j, 1.05 + 1.0j]
la, lb, lc, ld = mobius_from_three(source_pts, target_pts)
image_targets = [mobius(z, la, lb, lc, ld) for z in source_pts]
correspondence_residual = max(abs(w - target) for w, target in zip(image_targets, target_pts))
lab_circle = 0.1 + 0.75 * np.exp(1j * theta)
lab_image = finite_mobius_array(lab_circle, la, lb, lc, ld, cutoff=7)
lab_image_finite = lab_image[np.isfinite(lab_image.real) & np.isfinite(lab_image.imag)]
_, _, lab_circle_residual = fit_circle(lab_image_finite[::4])
lab_sample = [-0.8 + 0.2j, -0.1 + 0.9j, 0.6 - 0.45j, 1.1 + 0.35j]
lab_image_sample = [mobius(z, la, lb, lc, ld) for z in lab_sample]
lab_cr_error = abs(cp1_cross_ratio(*lab_sample) - cp1_cross_ratio(*lab_image_sample))

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.8), constrained_layout=True)
ax = axes[0]
plot_complex_axes(ax, 1.8)
ax.plot(lab_circle.real, lab_circle.imag, color="#BAB0AC", lw=1.5)
ax.scatter([z.real for z in source_pts], [z.imag for z in source_pts], color="#4C78A8", s=48, label="source")
for idx, z in enumerate(source_pts, start=1):
    ax.text(z.real + 0.04, z.imag + 0.05, f"z{idx}")
ax.set_title("Three source points")
ax.legend(loc="upper right")

ax = axes[1]
plot_complex_axes(ax, 2.0)
ax.plot(lab_image.real, lab_image.imag, color="#E45756", lw=1.8)
ax.scatter([z.real for z in target_pts], [z.imag for z in target_pts], color="#59A14F", s=48, label="target")
for idx, z in enumerate(target_pts, start=1):
    ax.text(z.real + 0.04, z.imag + 0.05, f"w{idx}")
ax.set_title("Solved Mobius image")
ax.legend(loc="upper right")

path = record_artifact(figure_path("three-point-mobius-lab.png"), "image")
fig.savefig(path, bbox_inches="tight")
plt.close(fig)

checks["three_point_lab"] = {
    "correspondence_residual": float(correspondence_residual),
    "lab_cross_ratio_error": float(lab_cr_error),
    "lab_circle_fit_residual": float(lab_circle_residual),
    "matrix_determinant_abs": float(abs(la * ld - lb * lc)),
}
summary_rows = []
for section, values in checks.items():
    for key, value in values.items():
        if isinstance(value, (int, float, bool, str)):
            summary_rows.append({"section": section, "check": key, "value": value})
summary_table = record_artifact(save_table(summary_rows, ARTIFACT_ROOT, "tables", "chapter-17-check-summary.csv"), "table")
display_artifact(path, width=900)
display_artifact(summary_table)
checks["three_point_lab"]


## Artifact Gallery

These direct links keep the notebook readable before execution. The code cells above regenerate the same files under the chapter artifact subtree.

![CP1 two charts](../../artifacts/chapter-17-the-complex-projective-line/figures/cp1-two-charts-infinity.png)

![Cross-ratio tests](../../artifacts/chapter-17-the-complex-projective-line/figures/cross-ratio-circle-line-tests.png)

![Mobius grid action](../../artifacts/chapter-17-the-complex-projective-line/figures/mobius-grid-circle-action.png)

![Mobius reflections and inversion](../../artifacts/chapter-17-the-complex-projective-line/figures/mobius-reflection-inversion-sides.png)

![Grassmann-Plucker Ptolemy shadow](../../artifacts/chapter-17-the-complex-projective-line/figures/grassmann-plucker-ptolemy-shadow.png)

![Intersection angle](../../artifacts/chapter-17-the-complex-projective-line/figures/oriented-circle-intersection-angle.png)

![Stereographic projection](../../artifacts/chapter-17-the-complex-projective-line/figures/stereographic-projection-cp1-sphere.png)

![Three-point Mobius lab](../../artifacts/chapter-17-the-complex-projective-line/figures/three-point-mobius-lab.png)

Interactive artifacts: [Mobius grid action](../../artifacts/chapter-17-the-complex-projective-line/html/mobius-grid-circle-action.html), [stereographic projection](../../artifacts/chapter-17-the-complex-projective-line/html/stereographic-projection-cp1-sphere.html).

## Takeaways

- `CP1` is not just the complex plane with a label attached to infinity. It is a two-chart projective object, and homogeneous scaling by complex numbers is part of the representation.
- Cross-ratios turn the circle-or-line test into a projective condition. The imaginary part of a cross-ratio also records oriented sidedness.
- Mobius transformations preserve cross-ratios, generalized circles, and oriented circle sides. Anti-Mobius transformations preserve generalized circles but reverse oriented sidedness.
- The Grassmann-Plucker relation is the algebraic source of Ptolemy's theorem once bracket magnitudes are interpreted as Euclidean distances.
- The argument of a cross-ratio records the signed intersection angle of oriented circles, and Mobius transformations preserve it.
- Stereographic projection gives the global picture: `CP1` is a sphere, finite complex numbers are one chart, and infinity is the north pole.

In [ ]:
assert checks["cp1_charts"]["homogeneous_rescale_error"] < 1e-12
assert checks["property_tests"]["collinearity_quotient_imag"] < 1e-12
assert checks["property_tests"]["cocircular_cross_ratio_imag"] < 1e-12
assert checks["mobius_transform"]["cross_ratio_error"] < 1e-10
assert checks["mobius_transform"]["transformed_circle_fit_residual"] < 1e-8
assert checks["anti_mobius"]["conjugation_side_sum"] < 1e-12
assert checks["anti_mobius"]["inversion_side_sum"] < 1e-12
assert checks["anti_mobius"]["unit_circle_fixed_residual"] < 1e-12
assert checks["anti_mobius"]["inversion_involution_residual"] < 1e-12
assert checks["grassmann_plucker_ptolemy"]["sympy_identity_zero"]
assert checks["grassmann_plucker_ptolemy"]["numeric_gp_residual"] < 1e-12
assert checks["grassmann_plucker_ptolemy"]["rectangle_ptolemy_gap"] < 1e-12
assert checks["grassmann_plucker_ptolemy"]["noncyclic_ptolemy_gap_positive"] > 0
assert checks["intersection_angles"]["cross_ratio_tangent_angle_error"] < 1e-12
assert checks["intersection_angles"]["mobius_angle_invariance_error"] < 1e-12
assert checks["stereographic_projection"]["max_sphere_equation_residual"] < 1e-12
assert checks["stereographic_projection"]["max_inverse_roundtrip_residual"] < 1e-12
assert checks["stereographic_projection"]["large_value_distance_to_north_pole"] < 0.05
assert checks["three_point_lab"]["correspondence_residual"] < 1e-10
assert checks["three_point_lab"]["lab_cross_ratio_error"] < 1e-10
assert checks["three_point_lab"]["lab_circle_fit_residual"] < 1e-8

assert_artifacts([path for path in artifact_paths if path.suffix != ".json"], min_size=256)
image_stats = [image_stats_relative(path) for path in image_paths]
assert all(item["pixel_std"] > 2 for item in image_stats)
all_files_exist = all(path.exists() and path.stat().st_size > 256 for path in artifact_paths if path.suffix != ".json")

visual_checks = {
    "chapter": 17,
    "source_span": "sections 17.1-17.7; printed pages 311-328; PDF pages 333-350",
    "all_files_exist": bool(all_files_exist),
    "visual_count": len([p for p in artifact_paths if p.suffix.lower() in {".png", ".html", ".csv"}]),
    "raster_artifacts": image_stats,
    "html_artifacts": [rel(path) for path in html_paths],
    "table_artifact": rel(summary_table),
    "cross_ratio_error": checks["mobius_transform"]["cross_ratio_error"],
    "numeric_checks": {
        "complex_cross_ratio_residual": checks["mobius_transform"]["cross_ratio_error"],
        "mobius_circle_fit_residual": checks["mobius_transform"]["transformed_circle_fit_residual"],
        "anti_mobius_side_reversal": max(checks["anti_mobius"]["conjugation_side_sum"], checks["anti_mobius"]["inversion_side_sum"]),
        "ptolemy_equality_gap": checks["grassmann_plucker_ptolemy"]["rectangle_ptolemy_gap"],
        "intersection_angle_error": checks["intersection_angles"]["cross_ratio_tangent_angle_error"],
        "stereographic_sphere_residual": checks["stereographic_projection"]["max_sphere_equation_residual"],
        "three_point_lab_cross_ratio_error": checks["three_point_lab"]["lab_cross_ratio_error"],
    },
}
visual_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final = {
    "chapter": 17,
    "notebook": "part-03-measurements/chapter-17-the-complex-projective-line/17-the-complex-projective-line.ipynb",
    "source_span_used": "sections 17.1-17.7; printed pages 311-328; PDF pages 333-350",
    "artifacts": [rel(path) for path in artifact_paths if path.suffix.lower() in {".png", ".html", ".csv"}],
    "checks": checks,
    "visual_checks": rel(visual_path),
    "notebook_executed": True,
}
final_path = save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
final
